# Week 69, Day 4: Project: evaluator

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w069d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('tiny_runtime.py', 'w', encoding='utf-8') as f:
    f.write('class Environment:\n    def __init__(self, parent=None):\n        self.values, self.parent = {}, parent\n\n    def define(self, name, value):\n        self.values[name] = value\n\n    def find(self, name):\n        env = self\n        while env is not None:\n            if name in env.values:\n                return env\n            env = env.parent\n        return None\n\n    def get(self, name, line=None):\n        env = self.find(name)\n        if env is None:\n            raise TinyRuntimeError(f"undefined name {name!r}", line)\n        return env.values[name]\n\n    def assign(self, name, value, line=None):\n        env = self.find(name)\n        if env is None:\n            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)\n        env.values[name] = value\n\n\nclass ReturnSignal(Exception):\n    def __init__(self, value):\n        self.value = value\n\n\nclass TinyFunction:\n    def __init__(self, decl, closure):\n        self.decl, self.closure = decl, closure\n\n    def __repr__(self):\n        return f"<fn {self.decl.name}>"\n\n\nclass Builtin:\n    def __init__(self, name, arity, func):\n        self.name, self.arity, self.func = name, arity, func\n\n    def __repr__(self):\n        return f"<builtin {self.name}>"\n')

### Exercise: Step 16: environments

Variables live in **environments**, one per scope, each linked to the one around it. Write `Environment(parent=None)` with a dict `values`: `define(name, value)` creates or replaces a variable **in this scope**; `find(name)` returns the nearest environment (this one, or a parent, grandparent...) that has the name, or `None`; `get(name, line=None)` returns the value from the nearest scope that has it, raising `TinyRuntimeError(f"undefined name {name!r}", line)` otherwise; and `assign(name, value, line=None)` changes the variable **where it was defined**, raising `TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)` if there's no such variable.

In [ ]:
class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


class Environment:
    def __init__(self, parent=None):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_environments():\n    \"scopes\"\n    outer = Environment()\n    outer.define(\"x\", 1)\n    inner = Environment(outer)\n    inner.define(\"y\", 2)\n    assert inner.get(\"x\") == 1 and inner.get(\"y\") == 2 and inner.find(\"x\") is outer and inner.find(\"z\") is None\n    inner.assign(\"x\", 10)\n    assert outer.get(\"x\") == 10 and \"x\" not in inner.values\n    inner.define(\"x\", 99)\n    assert inner.get(\"x\") == 99 and outer.get(\"x\") == 10\n    for action, message in ((lambda: outer.get(\"y\", 4), \"undefined name 'y'\"),\n                            (lambda: inner.assign(\"z\", 1, 7), \"cannot assign to undefined name 'z'\")):\n        try:\n            action()\n            assert False\n        except TinyRuntimeError as e:\n            assert e.message == message\n    try:\n        outer.get(\"nope\", 4)\n    except TinyRuntimeError as e:\n        assert str(e) == \"line 4: undefined name 'nope'\"")

### Exercise: Step 17: values and truth

Tiny values are Python values underneath: numbers are `int` or `float`, strings `str`, `true`/`false` are `True`/`False`, `nil` is `None`. Write `truthy(value)` (only `nil` and `false` are false: `0` and `""` are true, unlike Python); `show(value)` (how `print` writes it: `nil`, `true`, `false`, whole floats without `.0` such as `6.0` as `6`, and anything else with `str`); `type_name(value)` (`"nil"`, `"bool"`, `"number"`, `"string"` or `"function"`); and `is_number(value)` (an `int` or `float` but not a `bool`).

In [ ]:
def truthy(value):
    ___

def show(value):
    ___

def type_name(value):
    ___

def is_number(value):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"Tiny's values\"\n    assert [truthy(v) for v in (None, False, 0, \"\", True, \"no\", 0.0)] == [False, False, True, True, True, True, True]\n    assert [show(v) for v in (None, True, False, 6.0, 3.5, 7, \"hi\", -2.0)] == [\"nil\", \"true\", \"false\", \"6\", \"3.5\", \"7\", \"hi\", \"-2\"]\n    assert [type_name(v) for v in (None, True, 3, 2.5, \"s\", print)] == [\"nil\", \"bool\", \"number\", \"number\", \"string\", \"function\"]\n    assert is_number(3) and is_number(1.5) and not is_number(True) and not is_number(\"3\")")

### Exercise: Step 18: evaluating expressions

Write the `Interpreter` methods that compute expressions (calls come in Step 20). `evaluate(node, env)` adds 1 to `self.steps` and returns: the value of a `Number`, `String` or `Literal`; a `Name`'s value from the environment (passing the node's line); for `Unary`, `-` of a number (otherwise `TinyRuntimeError(f"cannot negate {type}")`) or `not` of the operand's truth; for `Logical`, short-circuiting `and`/`or` that return one of the **operand values**, like Python's; and for `Binary`, the result of `binary(node, left, right)`. Write `binary`: `==` and `!=` compare type names and values (so `1 == true` is false); `+` joins two strings; otherwise both must be numbers (`TinyRuntimeError(f"cannot use {op} with {type} and {type}")` with the line); `/` and `%` by zero raise `"division by zero"`; `/` of two ints that divide exactly gives an `int` (`6 / 3` is `2`, `7 / 2` is `3.5`); and the rest work as in Python.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0

class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()


class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent

    def define(self, name, value):
        self.values[name] = value

    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None

    def get(self, name, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"undefined name {name!r}", line)
        return env.values[name]

    def assign(self, name, value, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)
        env.values[name] = value


class ReturnSignal(Exception):
    def __init__(self, value):
        self.value = value


class TinyFunction:
    def __init__(self, decl, closure):
        self.decl, self.closure = decl, closure

    def __repr__(self):
        return f"<fn {self.decl.name}>"


class Builtin:
    def __init__(self, name, arity, func):
        self.name, self.arity, self.func = name, arity, func

    def __repr__(self):
        return f"<builtin {self.name}>"

def truthy(value):
    return value is not None and value is not False


def show(value):
    if value is None:
        return "nil"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, float) and value.is_integer():
        return str(int(value)) if abs(value) < 1e16 else str(value)
    return str(value)


def type_name(value):
    if value is None:
        return "nil"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str):
        return "string"
    return "function"


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


class Interpreter:
    def __init__(self, output=None):
        self.output = [] if output is None else output
        self.globals = Environment()
        self.steps = 0

    def evaluate(self, node, env):
        ___

    def binary(self, node, left, right):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_expressions():\n    \"expressions\"\n    def ev(src, **variables):\n        interp = Interpreter()\n        env = Environment(interp.globals)\n        for k, v in variables.items():\n            env.define(k, v)\n        return interp.evaluate(Parser(tokenize(src)).expression(), env)\n    assert ev(\"1 + 2 * 3\") == 7 and ev(\"7 / 2\") == 3.5 and ev(\"6 / 3\") == 2 and type(ev(\"6 / 3\")) is int\n    assert ev(\"-x + 10 % 4\", x=3) == -1 and ev('\"ab\" + \"cd\"') == \"abcd\" and ev(\"1.5 * 2\") == 3.0\n    assert ev(\"1 == true\") is False and ev(\"nil == nil\") is True and ev('\"a\" != \"a\"') is False and ev(\"2 >= 2\") is True\n    assert ev(\"nil or 5\") == 5 and ev(\"0 and 9\") == 9 and ev(\"false and boom\") is False and ev(\"not 0\") is False\n    interp = Interpreter()\n    interp.evaluate(Parser(tokenize(\"1 + 2\")).expression(), interp.globals)\n    assert interp.steps == 3\n    for src, message in ((\"1 + \\\"a\\\"\", \"cannot use + with number and string\"), (\"-\\\"x\\\"\", \"cannot negate string\"),\n                         (\"5 % 0\", \"division by zero\"), (\"y\", \"undefined name 'y'\"), (\"true < 1\", \"cannot use < with bool and number\")):\n        try:\n            ev(src)\n            assert False, src\n        except TinyRuntimeError as e:\n            assert e.message == message and e.line == 1, e.message")

### Exercise: Step 19: running statements

Add statements to the interpreter (expressions are given). `execute(node, env)` adds 1 to `self.steps` and handles: `Let` (define in `env`), `Assign` (assign, passing the line), `Print` (append `show(value)` to `self.output`), `ExprStmt` (evaluate and discard), `Block` (run its statements in a **new** environment whose parent is `env`), `If` (run `then` if the test is truthy, else `otherwise` if there is one), and `While` (repeat the body while the test is truthy). Then `run_block(statements, env)` runs a list of statements in order, and `run(program)` runs a whole program in `self.globals` and returns `self.output`.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0

class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()


class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent

    def define(self, name, value):
        self.values[name] = value

    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None

    def get(self, name, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"undefined name {name!r}", line)
        return env.values[name]

    def assign(self, name, value, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)
        env.values[name] = value


class ReturnSignal(Exception):
    def __init__(self, value):
        self.value = value


class TinyFunction:
    def __init__(self, decl, closure):
        self.decl, self.closure = decl, closure

    def __repr__(self):
        return f"<fn {self.decl.name}>"


class Builtin:
    def __init__(self, name, arity, func):
        self.name, self.arity, self.func = name, arity, func

    def __repr__(self):
        return f"<builtin {self.name}>"

def truthy(value):
    return value is not None and value is not False


def show(value):
    if value is None:
        return "nil"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, float) and value.is_integer():
        return str(int(value)) if abs(value) < 1e16 else str(value)
    return str(value)


def type_name(value):
    if value is None:
        return "nil"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str):
        return "string"
    return "function"


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


class Interpreter:
    def __init__(self, output=None):
        self.output = [] if output is None else output
        self.globals = Environment()
        self.steps = 0

    def evaluate(self, node, env):
        self.steps += 1
        if isinstance(node, (Number, String, Literal)):
            return node.value
        if isinstance(node, Name):
            return env.get(node.name, node.line)
        if isinstance(node, Unary):
            value = self.evaluate(node.operand, env)
            if node.op == "-":
                if not is_number(value):
                    raise TinyRuntimeError(f"cannot negate {type_name(value)}", node.line)
                return -value
            return not truthy(value)
        if isinstance(node, Logical):
            left = self.evaluate(node.left, env)
            if node.op == "or":
                return left if truthy(left) else self.evaluate(node.right, env)
            return left if not truthy(left) else self.evaluate(node.right, env)
        if isinstance(node, Binary):
            return self.binary(node, self.evaluate(node.left, env), self.evaluate(node.right, env))
        raise TinyRuntimeError(f"cannot evaluate {type(node).__name__}", getattr(node, "line", None))

    def binary(self, node, left, right):
        op = node.op
        if op == "==":
            return type_name(left) == type_name(right) and left == right
        if op == "!=":
            return not (type_name(left) == type_name(right) and left == right)
        if op == "+" and isinstance(left, str) and isinstance(right, str):
            return left + right
        if not (is_number(left) and is_number(right)):
            raise TinyRuntimeError(
                f"cannot use {op} with {type_name(left)} and {type_name(right)}", node.line)
        if op in ("/", "%") and right == 0:
            raise TinyRuntimeError("division by zero", node.line)
        if op == "/":
            result = left / right
            return int(result) if isinstance(left, int) and isinstance(right, int) and result.is_integer() else result
        return {"+": lambda: left + right, "-": lambda: left - right, "*": lambda: left * right,
                "%": lambda: left % right, "<": lambda: left < right, "<=": lambda: left <= right,
                ">": lambda: left > right, ">=": lambda: left >= right}[op]()

    def execute(self, node, env):
        ___

    def run_block(self, statements, env):
        ___

    def run(self, program):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_statements():\n    \"statements and scope\"\n    def out(src):\n        return Interpreter().run(parse(src))\n    assert out(\"let x = 2; print x * 3; x = x + 1; print x;\") == [\"6\", \"3\"]\n    assert out(\"let x = 1; { let x = 2; print x; } print x;\") == [\"2\", \"1\"]\n    assert out(\"let x = 1; { x = 5; } print x;\") == [\"5\"]\n    assert out(\"let i = 0; while (i < 3) { print i; i = i + 1; }\") == [\"0\", \"1\", \"2\"]\n    assert out('if (0) print \"zero is true\"; else print \"never\";') == [\"zero is true\"]\n    assert out('if (nil) print 1; else if (false) print 2; else print 3;') == [\"3\"]\n    assert out(\"print 7 / 2; print 1 == 1.0; print nil;\") == [\"3.5\", \"true\", \"nil\"]\n    try:\n        out(\"{ let hidden = 1; } print hidden;\")\n        assert False\n    except TinyRuntimeError as e:\n        assert e.message == \"undefined name 'hidden'\"")

### Exercise: Step 20: functions and closures

Finish the interpreter. In `execute`, a `Function` statement defines its name as a `TinyFunction(node, env)`, capturing the environment it was defined in (that's what makes it a **closure**), and a `Return` raises `ReturnSignal` with its value (or `None`). Write `call(node, env)`: evaluate the callee and the arguments; for a `Builtin`, check the number of arguments and call its `func` (re-raising a `TinyRuntimeError` from it with the call's line); for a `TinyFunction`, check the number of arguments (`f"{name}() takes {n} argument(s), got {m}"`), refuse to go deeper than `self.max_depth` calls (`"too much recursion"`), run its body in a new environment whose **parent is the closure** with the parameters defined, and return the value of a `ReturnSignal`, or `None` if the body finishes without one; calling anything else raises `f"cannot call {type}"`. Finally, `run_source(source)`: parse and run a program, returning its output. The rest is given.

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0

class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()

class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent

    def define(self, name, value):
        self.values[name] = value

    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None

    def get(self, name, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"undefined name {name!r}", line)
        return env.values[name]

    def assign(self, name, value, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)
        env.values[name] = value


class ReturnSignal(Exception):
    def __init__(self, value):
        self.value = value


class TinyFunction:
    def __init__(self, decl, closure):
        self.decl, self.closure = decl, closure

    def __repr__(self):
        return f"<fn {self.decl.name}>"


class Builtin:
    def __init__(self, name, arity, func):
        self.name, self.arity, self.func = name, arity, func

    def __repr__(self):
        return f"<builtin {self.name}>"

def truthy(value):
    return value is not None and value is not False


def show(value):
    if value is None:
        return "nil"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, float) and value.is_integer():
        return str(int(value)) if abs(value) < 1e16 else str(value)
    return str(value)


def type_name(value):
    if value is None:
        return "nil"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str):
        return "string"
    return "function"


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


class Interpreter:
    def __init__(self, output=None, max_depth=100):
        self.output = [] if output is None else output
        self.globals = Environment()
        self.max_depth, self.depth = max_depth, 0
        self.steps = 0
        for name, arity, func in [
            ("len", 1, self._len), ("str", 1, lambda v: show(v)), ("type", 1, type_name),
        ]:
            self.globals.define(name, Builtin(name, arity, func))

    @staticmethod
    def _len(value):
        if not isinstance(value, str):
            raise TinyRuntimeError(f"len() needs a string, not {type_name(value)}")
        return len(value)

    def evaluate(self, node, env):
        self.steps += 1
        if isinstance(node, (Number, String, Literal)):
            return node.value
        if isinstance(node, Name):
            return env.get(node.name, node.line)
        if isinstance(node, Unary):
            value = self.evaluate(node.operand, env)
            if node.op == "-":
                if not is_number(value):
                    raise TinyRuntimeError(f"cannot negate {type_name(value)}", node.line)
                return -value
            return not truthy(value)
        if isinstance(node, Logical):
            left = self.evaluate(node.left, env)
            if node.op == "or":
                return left if truthy(left) else self.evaluate(node.right, env)
            return left if not truthy(left) else self.evaluate(node.right, env)
        if isinstance(node, Binary):
            return self.binary(node, self.evaluate(node.left, env), self.evaluate(node.right, env))
        if isinstance(node, Call):
            return self.call(node, env)
        raise TinyRuntimeError(f"cannot evaluate {type(node).__name__}", getattr(node, "line", None))

    def binary(self, node, left, right):
        op = node.op
        if op == "==":
            return type_name(left) == type_name(right) and left == right
        if op == "!=":
            return not (type_name(left) == type_name(right) and left == right)
        if op == "+" and isinstance(left, str) and isinstance(right, str):
            return left + right
        if not (is_number(left) and is_number(right)):
            raise TinyRuntimeError(
                f"cannot use {op} with {type_name(left)} and {type_name(right)}", node.line)
        if op in ("/", "%") and right == 0:
            raise TinyRuntimeError("division by zero", node.line)
        if op == "/":
            result = left / right
            return int(result) if isinstance(left, int) and isinstance(right, int) and result.is_integer() else result
        return {"+": lambda: left + right, "-": lambda: left - right, "*": lambda: left * right,
                "%": lambda: left % right, "<": lambda: left < right, "<=": lambda: left <= right,
                ">": lambda: left > right, ">=": lambda: left >= right}[op]()

    def call(self, node, env):
        ___

    def execute(self, node, env):
        self.steps += 1
        if isinstance(node, Let):
            env.define(node.name, self.evaluate(node.value, env))
        elif isinstance(node, Assign):
            env.assign(node.name, self.evaluate(node.value, env), node.line)
        elif isinstance(node, Print):
            self.output.append(show(self.evaluate(node.value, env)))
        elif isinstance(node, ExprStmt):
            self.evaluate(node.value, env)
        elif isinstance(node, Block):
            self.run_block(node.body, Environment(env))
        elif isinstance(node, If):
            if truthy(self.evaluate(node.test, env)):
                self.execute(node.then, env)
            elif node.otherwise is not None:
                self.execute(node.otherwise, env)
        elif isinstance(node, While):
            while truthy(self.evaluate(node.test, env)):
                self.execute(node.body, env)
        # Step 20: Function and Return go here
        ___
        else:
            raise TinyRuntimeError(f"cannot execute {type(node).__name__}", getattr(node, "line", None))

    def run_block(self, statements, env):
        for stmt in statements:
            self.execute(stmt, env)

    def run(self, program):
        try:
            self.run_block(program, self.globals)
        except ReturnSignal:
            raise TinyRuntimeError("return outside a function") from None
        return self.output


def run_source(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_functions():\n    \"functions, closures and built-ins\"\n    assert run_source(\"fn add(a, b) { return a + b; } print add(2, 3);\") == [\"5\"]\n    assert run_source(\"fn fib(n) { if (n < 2) return n; return fib(n - 1) + fib(n - 2); } print fib(15);\") == [\"610\"]\n    counter = \"\"\"\nfn make_counter() {\n    let n = 0;\n    fn step() { n = n + 1; return n; }\n    return step;\n}\nlet a = make_counter();\nlet b = make_counter();\na(); a();\nprint a(); print b();\n\"\"\"\n    assert run_source(counter) == [\"3\", \"1\"]\n    assert run_source('fn hi() { print \"hi\"; } print hi();') == [\"hi\", \"nil\"]\n    assert run_source('print len(\"tiny\"); print str(2.0) + \"!\"; print type(len);') == [\"4\", \"2!\", \"function\"]\n    for src, message in ((\"fn f(x) { return x; } f();\", \"f() takes 1 argument(s), got 0\"),\n                         (\"let x = 3; x();\", \"cannot call number\"),\n                         (\"fn f() { return f(); } f();\", \"too much recursion\"),\n                         (\"len(5);\", \"len() needs a string, not number\"),\n                         (\"return 1;\", \"return outside a function\")):\n        try:\n            run_source(src)\n            assert False, src\n        except TinyRuntimeError as e:\n            assert e.message == message, e.message")

In [ ]:
from dataclasses import dataclass


class TinyError(Exception):
    """Base class for every error the Tiny interpreter reports."""

    def __init__(self, message, line=None, column=None):
        super().__init__(message)
        self.message, self.line, self.column = message, line, column

    def __str__(self):
        if self.line is None:
            return self.message
        if self.column is None:
            return f"line {self.line}: {self.message}"
        return f"line {self.line}, column {self.column}: {self.message}"


class LexError(TinyError):
    """The source contains characters that don't form a token."""


class ParseError(TinyError):
    """The tokens don't follow Tiny's grammar."""


class TinyRuntimeError(TinyError):
    """Something went wrong while a program ran."""


@dataclass(frozen=True)
class Token:
    kind: str
    text: str
    value: object
    line: int
    column: int


class Cursor:
    def __init__(self, source):
        self.source, self.pos, self.line, self.column = source, 0, 1, 1

    def at_end(self):
        return self.pos >= len(self.source)

    def peek(self, ahead=0):
        i = self.pos + ahead
        return self.source[i] if i < len(self.source) else ""

    def advance(self):
        ch = self.source[self.pos]
        self.pos += 1
        if ch == "\n":
            self.line, self.column = self.line + 1, 1
        else:
            self.column += 1
        return ch


def read_number(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isdigit():
        text += cursor.advance()
    if cursor.peek() == "." and cursor.peek(1).isdigit():
        text += cursor.advance()
        while cursor.peek().isdigit():
            text += cursor.advance()
        return Token("NUMBER", text, float(text), line, column)
    return Token("NUMBER", text, int(text), line, column)


ESCAPES = {"n": "\n", "t": "\t", '"': '"', "\\": "\\"}


def read_string(cursor):
    line, column = cursor.line, cursor.column
    text = cursor.advance()
    value = ""
    while True:
        if cursor.at_end() or cursor.peek() == "\n":
            raise LexError("unterminated string", line, column)
        ch = cursor.advance()
        text += ch
        if ch == '"':
            return Token("STRING", text, value, line, column)
        if ch == "\\":
            if cursor.at_end():
                raise LexError("unterminated string", line, column)
            esc_line, esc_col = cursor.line, cursor.column - 1
            code = cursor.advance()
            text += code
            if code not in ESCAPES:
                raise LexError(f"unknown escape \\{code}", esc_line, esc_col)
            value += ESCAPES[code]
        else:
            value += ch


KEYWORDS = {"let", "fn", "return", "if", "else", "while", "print", "true", "false", "nil", "and", "or", "not"}


TWO_CHAR = {"==", "!=", "<=", ">="}


ONE_CHAR = set("+-*/%<>=(){},;")


def read_name(cursor):
    line, column, text = cursor.line, cursor.column, ""
    while cursor.peek().isalnum() or cursor.peek() == "_":
        text += cursor.advance()
    if text in KEYWORDS:
        value = {"true": True, "false": False, "nil": None}.get(text)
        return Token(text.upper(), text, value, line, column)
    return Token("NAME", text, text, line, column)


def read_operator(cursor):
    line, column = cursor.line, cursor.column
    pair = cursor.peek() + cursor.peek(1)
    if pair in TWO_CHAR:
        cursor.advance()
        cursor.advance()
        return Token("OP", pair, pair, line, column)
    ch = cursor.peek()
    if ch in ONE_CHAR:
        cursor.advance()
        return Token("OP", ch, ch, line, column)
    raise LexError(f"unexpected character {ch!r}", line, column)


def skip_space_and_comments(cursor):
    while not cursor.at_end():
        ch = cursor.peek()
        if ch in " \t\r\n":
            cursor.advance()
        elif ch == "#":
            while not cursor.at_end() and cursor.peek() != "\n":
                cursor.advance()
        else:
            return


def tokenize(source):
    cursor = Cursor(source)
    tokens = []
    while True:
        skip_space_and_comments(cursor)
        if cursor.at_end():
            tokens.append(Token("EOF", "", None, cursor.line, cursor.column))
            return tokens
        ch = cursor.peek()
        if ch.isdigit():
            tokens.append(read_number(cursor))
        elif ch == '"':
            tokens.append(read_string(cursor))
        elif ch.isalpha() or ch == "_":
            tokens.append(read_name(cursor))
        else:
            tokens.append(read_operator(cursor))

@dataclass
class Number:
    value: object
    line: int = 0


@dataclass
class String:
    value: str
    line: int = 0


@dataclass
class Literal:
    value: object
    line: int = 0


@dataclass
class Name:
    name: str
    line: int = 0


@dataclass
class Unary:
    op: str
    operand: object
    line: int = 0


@dataclass
class Binary:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Logical:
    op: str
    left: object
    right: object
    line: int = 0


@dataclass
class Call:
    callee: object
    args: list
    line: int = 0


@dataclass
class Let:
    name: str
    value: object
    line: int = 0


@dataclass
class Assign:
    name: str
    value: object
    line: int = 0


@dataclass
class Print:
    value: object
    line: int = 0


@dataclass
class ExprStmt:
    value: object
    line: int = 0


@dataclass
class Block:
    body: list
    line: int = 0


@dataclass
class If:
    test: object
    then: object
    otherwise: object
    line: int = 0


@dataclass
class While:
    test: object
    body: object
    line: int = 0


@dataclass
class Function:
    name: str
    params: list
    body: list
    line: int = 0


@dataclass
class Return:
    value: object
    line: int = 0

class Parser:
    def __init__(self, tokens):
        self.tokens, self.pos = tokens, 0

    def peek(self):
        return self.tokens[self.pos]

    def at(self, kind, text=None):
        tok = self.peek()
        return tok.kind == kind and (text is None or tok.text == text)

    def match(self, kind, text=None):
        if self.at(kind, text):
            tok = self.peek()
            self.pos += 1
            return tok
        return None

    def expect(self, kind, text=None, what=None):
        tok = self.match(kind, text)
        if tok is None:
            got = self.peek()
            shown = "end of input" if got.kind == "EOF" else repr(got.text)
            raise ParseError(f"expected {what or text or kind.lower()}, got {shown}", got.line, got.column)
        return tok

    # --- expressions, lowest precedence first ---
    def expression(self):
        return self.logic_or()

    def logic_or(self):
        node = self.logic_and()
        while self.at("OR"):
            line = self.match("OR").line
            node = Logical("or", node, self.logic_and(), line)
        return node

    def logic_and(self):
        node = self.equality()
        while self.at("AND"):
            line = self.match("AND").line
            node = Logical("and", node, self.equality(), line)
        return node

    def binary(self, operators, operand):
        node = operand()
        while self.peek().kind == "OP" and self.peek().text in operators:
            tok = self.match("OP")
            node = Binary(tok.text, node, operand(), tok.line)
        return node

    def equality(self):
        return self.binary({"==", "!="}, self.comparison)

    def comparison(self):
        return self.binary({"<", "<=", ">", ">="}, self.term)

    def term(self):
        return self.binary({"+", "-"}, self.factor)

    def factor(self):
        return self.binary({"*", "/", "%"}, self.unary)

    def unary(self):
        if self.at("OP", "-"):
            line = self.match("OP").line
            return Unary("-", self.unary(), line)
        if self.at("NOT"):
            line = self.match("NOT").line
            return Unary("not", self.unary(), line)
        return self.call()

    def call(self):
        node = self.primary()
        while self.at("OP", "("):
            line = self.match("OP").line
            args = []
            if not self.at("OP", ")"):
                args.append(self.expression())
                while self.match("OP", ","):
                    args.append(self.expression())
            self.expect("OP", ")")
            node = Call(node, args, line)
        return node

    def primary(self):
        tok = self.peek()
        if self.match("NUMBER"):
            return Number(tok.value, tok.line)
        if self.match("STRING"):
            return String(tok.value, tok.line)
        if tok.kind in ("TRUE", "FALSE", "NIL"):
            self.pos += 1
            return Literal(tok.value, tok.line)
        if self.match("NAME"):
            return Name(tok.text, tok.line)
        if self.match("OP", "("):
            node = self.expression()
            self.expect("OP", ")")
            return node
        shown = "end of input" if tok.kind == "EOF" else repr(tok.text)
        raise ParseError(f"expected an expression, got {shown}", tok.line, tok.column)

    # --- statements ---
    def statement(self):
        tok = self.peek()
        if self.match("LET"):
            name = self.expect("NAME", what="a name").text
            self.expect("OP", "=")
            value = self.expression()
            self.expect("OP", ";")
            return Let(name, value, tok.line)
        if self.match("PRINT"):
            value = self.expression()
            self.expect("OP", ";")
            return Print(value, tok.line)
        if self.at("OP", "{"):
            return Block(self.block(), tok.line)
        if self.match("IF"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            then = self.statement()
            otherwise = self.statement() if self.match("ELSE") else None
            return If(test, then, otherwise, tok.line)
        if self.match("WHILE"):
            self.expect("OP", "(")
            test = self.expression()
            self.expect("OP", ")")
            return While(test, self.statement(), tok.line)
        if self.match("FN"):
            name = self.expect("NAME", what="a function name").text
            self.expect("OP", "(")
            params = []
            if not self.at("OP", ")"):
                params.append(self.expect("NAME", what="a parameter name").text)
                while self.match("OP", ","):
                    params.append(self.expect("NAME", what="a parameter name").text)
            self.expect("OP", ")")
            return Function(name, params, self.block(), tok.line)
        if self.match("RETURN"):
            value = None if self.at("OP", ";") else self.expression()
            self.expect("OP", ";")
            return Return(value, tok.line)
        if tok.kind == "NAME" and self.tokens[self.pos + 1].kind == "OP" and self.tokens[self.pos + 1].text == "=":
            self.pos += 2
            value = self.expression()
            self.expect("OP", ";")
            return Assign(tok.text, value, tok.line)
        value = self.expression()
        self.expect("OP", ";")
        return ExprStmt(value, tok.line)

    def block(self):
        self.expect("OP", "{")
        body = []
        while not self.at("OP", "}"):
            if self.at("EOF"):
                tok = self.peek()
                raise ParseError("expected '}', got end of input", tok.line, tok.column)
            body.append(self.statement())
        self.expect("OP", "}")
        return body

    def program(self):
        body = []
        while not self.at("EOF"):
            body.append(self.statement())
        return body


def parse(source):
    return Parser(tokenize(source)).program()

class Environment:
    def __init__(self, parent=None):
        self.values, self.parent = {}, parent

    def define(self, name, value):
        self.values[name] = value

    def find(self, name):
        env = self
        while env is not None:
            if name in env.values:
                return env
            env = env.parent
        return None

    def get(self, name, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"undefined name {name!r}", line)
        return env.values[name]

    def assign(self, name, value, line=None):
        env = self.find(name)
        if env is None:
            raise TinyRuntimeError(f"cannot assign to undefined name {name!r}", line)
        env.values[name] = value


class ReturnSignal(Exception):
    def __init__(self, value):
        self.value = value


class TinyFunction:
    def __init__(self, decl, closure):
        self.decl, self.closure = decl, closure

    def __repr__(self):
        return f"<fn {self.decl.name}>"


class Builtin:
    def __init__(self, name, arity, func):
        self.name, self.arity, self.func = name, arity, func

    def __repr__(self):
        return f"<builtin {self.name}>"

def truthy(value):
    return value is not None and value is not False


def show(value):
    if value is None:
        return "nil"
    if value is True:
        return "true"
    if value is False:
        return "false"
    if isinstance(value, float) and value.is_integer():
        return str(int(value)) if abs(value) < 1e16 else str(value)
    return str(value)


def type_name(value):
    if value is None:
        return "nil"
    if isinstance(value, bool):
        return "bool"
    if isinstance(value, (int, float)):
        return "number"
    if isinstance(value, str):
        return "string"
    return "function"


def is_number(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool)


class Interpreter:
    def __init__(self, output=None, max_depth=100):
        self.output = [] if output is None else output
        self.globals = Environment()
        self.max_depth, self.depth = max_depth, 0
        self.steps = 0
        for name, arity, func in [
            ("len", 1, self._len), ("str", 1, lambda v: show(v)), ("type", 1, type_name),
        ]:
            self.globals.define(name, Builtin(name, arity, func))

    @staticmethod
    def _len(value):
        if not isinstance(value, str):
            raise TinyRuntimeError(f"len() needs a string, not {type_name(value)}")
        return len(value)

    # --- expressions ---
    def evaluate(self, node, env):
        self.steps += 1
        if isinstance(node, (Number, String, Literal)):
            return node.value
        if isinstance(node, Name):
            return env.get(node.name, node.line)
        if isinstance(node, Unary):
            value = self.evaluate(node.operand, env)
            if node.op == "-":
                if not is_number(value):
                    raise TinyRuntimeError(f"cannot negate {type_name(value)}", node.line)
                return -value
            return not truthy(value)
        if isinstance(node, Logical):
            left = self.evaluate(node.left, env)
            if node.op == "or":
                return left if truthy(left) else self.evaluate(node.right, env)
            return left if not truthy(left) else self.evaluate(node.right, env)
        if isinstance(node, Binary):
            return self.binary(node, self.evaluate(node.left, env), self.evaluate(node.right, env))
        if isinstance(node, Call):
            return self.call(node, env)
        raise TinyRuntimeError(f"cannot evaluate {type(node).__name__}", getattr(node, "line", None))

    def binary(self, node, left, right):
        op = node.op
        if op == "==":
            return type_name(left) == type_name(right) and left == right
        if op == "!=":
            return not (type_name(left) == type_name(right) and left == right)
        if op == "+" and isinstance(left, str) and isinstance(right, str):
            return left + right
        if not (is_number(left) and is_number(right)):
            raise TinyRuntimeError(
                f"cannot use {op} with {type_name(left)} and {type_name(right)}", node.line)
        if op in ("/", "%") and right == 0:
            raise TinyRuntimeError("division by zero", node.line)
        if op == "/":
            result = left / right
            return int(result) if isinstance(left, int) and isinstance(right, int) and result.is_integer() else result
        return {"+": lambda: left + right, "-": lambda: left - right, "*": lambda: left * right,
                "%": lambda: left % right, "<": lambda: left < right, "<=": lambda: left <= right,
                ">": lambda: left > right, ">=": lambda: left >= right}[op]()

    def call(self, node, env):
        callee = self.evaluate(node.callee, env)
        args = [self.evaluate(a, env) for a in node.args]
        if isinstance(callee, Builtin):
            if len(args) != callee.arity:
                raise TinyRuntimeError(f"{callee.name}() takes {callee.arity} argument(s), got {len(args)}", node.line)
            try:
                return callee.func(*args)
            except TinyRuntimeError as e:
                raise TinyRuntimeError(e.message, node.line) from None
        if not isinstance(callee, TinyFunction):
            raise TinyRuntimeError(f"cannot call {type_name(callee)}", node.line)
        decl = callee.decl
        if len(args) != len(decl.params):
            raise TinyRuntimeError(f"{decl.name}() takes {len(decl.params)} argument(s), got {len(args)}", node.line)
        if self.depth >= self.max_depth:
            raise TinyRuntimeError("too much recursion", node.line)
        local = Environment(callee.closure)
        for name, value in zip(decl.params, args):
            local.define(name, value)
        self.depth += 1
        try:
            self.run_block(decl.body, local)
        except ReturnSignal as signal:
            return signal.value
        finally:
            self.depth -= 1
        return None

    # --- statements ---
    def execute(self, node, env):
        self.steps += 1
        if isinstance(node, Let):
            env.define(node.name, self.evaluate(node.value, env))
        elif isinstance(node, Assign):
            env.assign(node.name, self.evaluate(node.value, env), node.line)
        elif isinstance(node, Print):
            self.output.append(show(self.evaluate(node.value, env)))
        elif isinstance(node, ExprStmt):
            self.evaluate(node.value, env)
        elif isinstance(node, Block):
            self.run_block(node.body, Environment(env))
        elif isinstance(node, If):
            if truthy(self.evaluate(node.test, env)):
                self.execute(node.then, env)
            elif node.otherwise is not None:
                self.execute(node.otherwise, env)
        elif isinstance(node, While):
            while truthy(self.evaluate(node.test, env)):
                self.execute(node.body, env)
        elif isinstance(node, Function):
            env.define(node.name, TinyFunction(node, env))
        elif isinstance(node, Return):
            raise ReturnSignal(None if node.value is None else self.evaluate(node.value, env))
        else:
            raise TinyRuntimeError(f"cannot execute {type(node).__name__}", getattr(node, "line", None))

    def run_block(self, statements, env):
        for stmt in statements:
            self.execute(stmt, env)

    def run(self, program):
        try:
            self.run_block(program, self.globals)
        except ReturnSignal:
            raise TinyRuntimeError("return outside a function") from None
        return self.output


def run_source(source):
    interpreter = Interpreter()
    interpreter.run(parse(source))
    return interpreter.output

def show_error(source, error):
    if error.line is None:
        return str(error)
    text = source.split("\n")[error.line - 1]
    if error.column is None:
        return f"{error}\n    {text}"
    return f"{error}\n    {text}\n    {' ' * (error.column - 1)}^"

program = """
fn make_counter() {
    let count = 0;
    fn step() {
        count = count + 1;
        return count;
    }
    return step;
}
let next = make_counter();
next();
next();
print next();
fn fib(n) {
    if (n < 2) return n;
    return fib(n - 1) + fib(n - 2);
}
print "fib(15) is " + str(fib(15));
let i = 0;
while (i < 3) {
    print i * i;
    i = i + 1;
}
"""
for line in run_source(program):
    print(line)

broken = "let total = 0;\nprint total + \"one\";"
try:
    run_source(broken)
except TinyError as e:
    print(show_error(broken, e))

> **Closures, from the inside.** Week 60 explained closures from the outside: a nested function remembers variables from the function around it. Now you've built one. The whole trick is a single line: `TinyFunction(node, env)` stores the environment that was current when `fn` ran, and every call makes its new scope a child of **that** environment, not of the caller's. So `step` finds `count` in the environment of the `make_counter` call that created it, even after that call has returned, and each counter has its own `count` because each call to `make_counter` made a new environment. Python's cells and `nonlocal` (Week 60) are a faster version of the same idea.

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Where we are

The front end is finished: `parse(source)` gives a list of statement nodes. The evaluator needs four small classes besides itself, given here so the exercises can concentrate on the logic:

A `TinyFunction` is a function's declaration plus the environment it was created in. A `Builtin` wraps a Python function so Tiny can call it. And `ReturnSignal` is an exception used for something that isn't an error: a `return` statement deep inside nested blocks and loops must jump straight out to the function call, and raising an exception is the simplest way to unwind all the Python calls in between.

## Step 16: environments

## Step 17: values and truth

## Step 18: evaluating expressions

The evaluator is one method with a branch per kind of node, each evaluating the node's children first. It's the same post-order walk as Week 64's compiler, but instead of emitting instructions, it computes values directly.

## Step 19: running statements

## Step 20: functions and closures

## Checkpoint

The whole interpreter, running Monday's example program:

## Quiz

## Before you go

Write a Tiny program of your own that uses a closure in a way the counter doesn't, such as a function that returns an "adder" (`make_adder(5)` returning a function that adds 5), and run it with today's interpreter. Tomorrow: testing the interpreter properly, and finding out where it spends its time.